# Ayudantía 10: Interfaces graficas 2 🎨
Por sus queridos ayudantes de catedra:

* Diego Toledo
* Francisca Cares
* Carlos Martel
* Agustín Becker
* Julio Huerta

# Diagrama de clase

Un diagrama de clases es una forma de representar de forma visual como se estructuran las clases de un código. Cada clase tiene:
* Nombre
* atributos 
* métodos.

Pero tambien incluye información sobre como se relacionan las distintas clases, aquí una explicación    

### Relaciones
Dentro de nuestro código se espera que las clases se relacionen ente ellas, dentro de los diagramas de clases se van a intentar representar estas relaciones con distintas simbologías.

* Herencia: Si una clase hereda de otra, se conectarán ambas con una linea y un triángulo. El triángulo apuntará siempre a la clase padre.
* Agregación: Un objeto A contiene a otro objeto B en su interior (en una lista, en un atributo etc.), pero cumple con que **B puede funcionar independientemente de A**. En el diagrama se representa con una línea y **un diamante blanco** cerca de la clase que contiene a la otra.
* Composición: Un objeto A contiene a otro objeto B en su interior (en una lista, en un atributo etc.), pero cumple con que **B solo existe dentro de A**. En el diagrama se representa con una línea y **un diamante negro** cerca de la clase que contiene a la otra.

Por ejemplo, si tengo un estuche el cual contiene diversos lapices en su interior. Si en el código mi estuche desaparece ¿Los lapices deben desaparecer también?. Si la respuesta es que no, tenemos **agregación**. Si la respuesta es si, estamos frente a una **composición**.

Recuerda que pese a que la agregación y la composición son elementos que proporcionan información importante y recomendamos utilizarlos en sus diagramas.


![Texto alternativo](image.png)





# Funcionalidades nuevas de interfaces gráficas

En las últimas semanas hemos podido ver muchos nuevos contenidos. Los más importantes siendo networking y threading, estos contenidos nos abren un mundo de posibilidades para que nuestros códigos tengan comportamientos más complejos. Por otro lado, aprendimos como utilizar PyQt5 para generar interfaces gráficas que permitieran que cualquier persona interaccionara con el código. Pero existe un problema...

## El problema

PyQt es una libreria que funciona muy distinto a como funciona python normalmente (Ya que PyQt5 tiene un arquitectura basada en eventos). Esto genera que cada vez que intentemos utilizar threading o funciones del código que detengan la ejecución momentaneamente se generarán errores o comportamientos inesperados.

Pese a esto, PyQt viene equipado con formas de evitar estas problemáticas. 

## Qthread

Debido a que los threads normales de python generan conflicto, pyqt cuenta con los el objeto Qthread. Los Qthread tienen un funcionamiento similar a los threads normales de python pero con pequeñas diferencias en lo métodos con los que cuentas.

Aquí una comparación:

In [ ]:

import threading
import time

def tarea_larga():
    for i in range(5):
        print(f"[threading] Contando: {i}")
        time.sleep(1)

# Crear e iniciar el hilo
hilo = threading.Thread(target=tarea_larga)

# Continuar con el hilo principal
print("[threading] Hilo lanzado")

hilo.start()


# Esperar a que termine
hilo.join()
print("[threading] Hilo terminado")


[threading] Contando: 0
[threading] Hilo lanzado
[threading] Contando: 1
[threading] Contando: 2
[threading] Contando: 3
[threading] Contando: 4
[threading] Hilo terminado


In [ ]:
from PyQt5.QtCore import QThread
import sys
import time
from PyQt5.QtWidgets import QApplication

class MiThread(QThread):
    def run(self):
        for i in range(5):
            print(f"[QThread] Contando: {i}")
            time.sleep(1)

app = QApplication(sys.argv)

hilo = MiThread()
hilo.start()

print("[QThread] Hilo lanzado")

# Para esperar que termine debemos usar el método wait en vez de join
hilo.wait()
print("[QThread] Hilo terminado")


[QThread] Hilo lanzado
[QThread] Contando: 0
[QThread] Contando: 1
[QThread] Contando: 2
[QThread] Contando: 3
[QThread] Contando: 4
[QThread] Hilo terminado


Son el mismo código, pero el segundo será efectivo cuando se quiera utiizar pyqt. Adicionalmente, debido a que al trabajar con concurrencia se tienen ciertas zonas críticas que queremos proteger para que 2 threads no puedan acceder al mismo tiempo. PyQt nos incluye un homologo al Lock de threading, el **Qmutex**-

| Característica                         | `threading.Lock` | `QMutex`       |
|----------------------------------------|-------------------------------------|--------------------------------------|
| Importación                            | `from threading import Lock`        | `from PyQt5.QtCore import QMutex`    |
| Crear instancia                        | `lock = Lock()`                     | `mutex = QMutex()`                   |
| Bloquear manualmente                   | `lock.acquire()`                    | `mutex.lock()`                       |
| Liberar bloqueo                        | `lock.release()`                    | `mutex.unlock()`                     |
| Uso con `with`                         | ✅ `with lock:`                      | ❌ No se puede utilizar with          |
| Ver si se puede bloquear sin esperar   | `lock.acquire(blocking=False)`      | `mutex.tryLock()`                    |


## Audio en PyQt5

PyQt5 tambien tiene la opción de reproducir audio. Para eso se utilizará el siguiente codigo:

**Advertencia** reproducir este codigo genera problemas debido a que PyQt no está adaptado para funcionar en un Jupyter

In [ ]:
from PyQt5.QtCore import QUrl
from PyQt5.QtMultimedia import QMediaPlayer, QMediaContent
import os
from os.path import join
from PyQt5.QtWidgets import QApplication
import sys

app = QApplication(sys.argv)

# -----------------LO IMPORTANTE --------------------------
# Creamos el objeto que reproduce mp3
sonido_perder = QMediaPlayer()

# Definimos la ruta al archivo de audio y le asigamos el audio al QMediaContent
path = os.path.abspath(join("sound", "die.mp3"))
content = QMediaContent(QUrl.fromLocalFile(path))
sonido_perder.setMedia(content)

# Finalmente reproducimos el audio
sonido_perder.play()
# ----------------------------------------------------------

sys.exit(app.exec_())


## PyQt y Networking

Debido a que tanto los sockets, como las peticiones mediante la libreria request **detienen el flujo del codigo** esperando una respuesta, generan que el código de PyQt se detenga tambien y la pantalla se queda congelada 😢. Es por esto que se debe asignar un Qthread a cada request.

Además, recordemos que el frontent **solo se encarga de la interacción con el usuario**. Si consideramos que la conexión con el servidor no es una interacción con el usuario, concluimos que **EL FRONTENT NO DEBE ENCARGARSE DE REALIZAR EL NETWORKING** (en este curso). Esa labor la debe realizar siempre el Backend.



# Ejercicio

En la carpeta eduroam_simulator encontrarán un código que contiene tanto un servidor como un juego.

* **Servidor:** El servidor es un API http encargada de llevar el conteo de puntos obtenidos por cada usuario. Tiene solamente un GET el cual se encarga de devolver todos los puntajes almacenados, y un método POST que permite publicar un nuevo puntaje. Este archivo debe ejecutarse antes de iniciar el juego.

* **Juego:** Contiene un archivo main.py el cual se encarga de iniciar tanto el backend como el frontent.



# Frontent


### ✅ Parte 1) Iniciar audios (Método: MiVentana.iniciar_sonidos)

Para mejorar la experiencia de nuestros jugadores ~~y que la copia sea más descarada~~ debemos cargar los audios para que estén listos cuando se decidan reproducir. Los audios que se deben preparar están dentro de la carpeta sound y tienen los nombres de "die.mp3","jump.mp3","point.mp3". Esto se realiza dentro del método iniciar_sonidos de la clase MiVentana.

### ✅ Parte 2) Reproducir audio (Método: MiVentana.sumar_puntaje)

Ya que cargamos los audios, debemos reproducirlos. Para esto se utiliza el metodo .play() sobre el QMediaContent. Un ejemplo de esto se encuentra en el método sumar_puntaje() de MiVentana.

Cada vez que el puntaje sea un múltiple de 5000 reproducimos un audio.

### ✅ Parte 3) Crear QTimer (Método: MiVentana.iniciar_threads)

Dentro de nuestro juego existen varias acciones que deben suceder de forma simultanea.

Por ejemplo:
* Constantemente se van generando Gatos nuevos.
* Constantemente se sube el puntaje obtenido.
* Constantemente se debe comprobar si el usuario ha tocado un gato.
* Constantemente el dinosaurio cambia su imagen para mover las patitas
* Constantemente cada gato individualmente va moviendose hacia la izquierda
* Etc.

Es por esto, que debemos utilizar threading. PyQt5 cuenta con un objeto similar a Qthread que se llama Qtimer. Este thread tiene la particularidad de que reproduce la función objetivo constantemente cada cierta cantidad de milisegundos que nosotrso definimos. Para detener la reproducción del thread se utiliza el método .stop() de los .

Para ver la implementación de estas cosas ir al método iniciar_thread de MiVentana.

### ✅ Parte 4) Implementar QMutex (Método: Dino.saltar)
Ahora que logramos implementar threading, debemos tener en cuenta qué partes del código son aquellas que pueden generar conflicto si 2 threads actuan al mismo tiempo.

El salto del dinosaurio es controlado por un Qtimer. El problema es que si alguien apreta espacio mientras el dinosaurio ya está saltando, 2 threads llegarán a intentar subir y bajar el dinosario al mismo tiempo 🤯. Esto lleva a que el dinosaurio pueda empezar a volar.

Por esto, el interactuar con la posicion del Dinosaurio es una **Sección crítica**. Por eso se le adjunta un QMutex.


In [ ]:
import sys
from os.path import join
import os

from PyQt5.QtWidgets import QWidget, QLabel, QLineEdit, QPushButton
from PyQt5.QtGui import QPixmap, QTransform, QKeyEvent
from PyQt5.QtCore import Qt, pyqtSignal, QTimer, QMutex, pyqtSignal, QUrl
from PyQt5.QtMultimedia import QMediaPlayer, QMediaContent

class MiVentana(QWidget):
    senal_obtener_clasificador = pyqtSignal()
    senal_nombre = pyqtSignal(str)
    senal_publicar_puntaje = pyqtSignal(int)

    def __init__(self) -> None:
        super().__init__()
        
        self.gatos = []
        self.seguimos = False

        self.iniciar_gui()
        self.iniciar_sonidos()
        self.iniciar_threads()

        self.setFocusPolicy(Qt.StrongFocus)
        self.setFocus()

    def iniciar_threads(self):

        # Qthread que ve si perdemos
        self.timer_colision = QTimer()
        self.timer_colision.timeout.connect(self.revisar_colisiones)
        self.timer_colision.start(100)  # cada 100 ms

        # Thread que crea un nuevo gato cada 2 segundos
        self.mas_gatos()
        self.timer_mas_gatos = QTimer()
        self.timer_mas_gatos.timeout.connect(self.mas_gatos)
        self.timer_mas_gatos.start(2000)

        # Thread encargado de subir el puntaje
        self.timer_puntaje = QTimer()
        self.timer_puntaje.timeout.connect(self.sumar_puntaje)
        self.timer_puntaje.start(100)


        #Thread que consulta el clasificador
        # Se deja en un thread aparte debido a que request detiene todo, osea que 
        # Congelaria el código

        self.timer_mas_gatos = QTimer()
        self.timer_mas_gatos.singleShot(1000, self.consultar_clasificador)

    def iniciar_sonidos(self):

        # ------------------PRECAUCIÓN------------------------------
        # Notar que se utiliza QMediaPlayer Solamente porque son archivos .mp3
        # En caso de ser .wav se deberia utilizar QSoundEffect


        # Sonido de perder
        self.sonido_perder = QMediaPlayer(self)
        path = os.path.abspath(join("frontent", "sound", "die.mp3"))
        content = QMediaContent(QUrl.fromLocalFile(path))
        self.sonido_perder.setMedia(content)

        # Sonido de saltar
        self.sonido_saltar = QMediaPlayer(self)
        path = os.path.abspath(join("frontent", "sound", "jump.mp3"))
        content = QMediaContent(QUrl.fromLocalFile(path))
        self.sonido_saltar.setMedia(content)

        # Sonido Puntos
        self.sonido_puntos = QMediaPlayer(self)
        path = os.path.abspath(join("frontent", "sound", "point.mp3"))
        content = QMediaContent(QUrl.fromLocalFile(path))
        self.sonido_puntos.setMedia(content)

    def iniciar_gui(self):

        self.setGeometry(400, 400, 2000, 800)

        #Creamos la linea inferior
        label = QLabel(self)
        label.setGeometry(0, 700, 1000000, 10)
        label.setStyleSheet("background-color: black;")

        #Creamos nueestro dinosaurio
        self.dinosaurio = Dino(self)


        # Creamos el campo para ingresar nombre
        self.nombre = QLineEdit("Nombre", self)
        self.nombre.setGeometry(700, 350, 800, 100)
        self.nombre.setAlignment(Qt.AlignCenter)

        # Creamos el boton para iniciar la partida
        self.boton = QPushButton("Iniciar",self)
        self.boton.setGeometry(950, 450, 300, 100)
        self.boton.clicked.connect(self.iniciar)

        #Ranking
        self.label_ranking = QLabel(self)
        self.label_ranking.setGeometry(1550, 50, 400, 600)
        self.label_ranking.setStyleSheet("font-size: 20px; background-color: #EEE; border: 1px solid black;")
        self.label_ranking.setAlignment(Qt.AlignTop)
        self.label_ranking.setWordWrap(True)

        # Qlabel de puntaje
        self.puntaje = 0
        self.label_puntaje = QLabel(f"Puntaje: {self.puntaje}", self)
        self.label_puntaje.setStyleSheet("font-size: 20px; background-color: #EEE; border: 1px solid black;")
        self.label_puntaje.setGeometry(10, 10, 400, 50)

    def sumar_puntaje(self):
        if self.seguimos:
            self.puntaje += 100
            self.label_puntaje.setText(f"Puntaje: {self.puntaje}")
            if self.puntaje % 5000 == 0:
                # Aquí reproducimos un sonido de puntos
                self.sonido_puntos.play()

    def consultar_clasificador(self):
        self.senal_obtener_clasificador.emit()

    def revisar_colisiones(self):
        if self.seguimos:
            for gato in self.gatos:
                if gato.geometry().intersects(self.dinosaurio.geometry()):
                    self.perder()

    def perder(self):
        # Hacemos que todo se detenga
        self.seguimos = False
        self.dinosaurio.seguimos = False
        for gato in self.gatos:
            gato.hide()

        # Volvemos a sacar el boton de iniciar
        self.boton.show()
        
        # Publicamos los nuevos puntajes
        self.senal_publicar_puntaje.emit(self.puntaje)

        # Detenemos la aparición de los gatos
        self.timer_mas_gatos.stop()

        # Reproducimos el sonido correspondiente
        self.sonido_perder.play()    

    def mas_gatos(self):
        if self.seguimos:
            gato = Gato(self)
            gato.show()
            gato.seguimos = True
            self.gatos.append(gato)

    def keyPressEvent(self, event: QKeyEvent) -> None:
        if event.key() == Qt.Key_Space:
            if self.seguimos:
                self.dinosaurio.saltar()
                self.sonido_saltar.play()

    def iniciar(self):
        self.puntaje = 0
        
        #Le decimos a todas las entidades que pueden iniciar su movimiento
        self.dinosaurio.seguimos = True
        self.seguimos = True
        self.gatos = []

        # Ocultamos la interfaz gráfica 
        self.boton.hide()
        self.nombre.hide()

        # Emito una señal al backend para que guarde el nombre
        self.senal_nombre.emit(self.nombre.text())
        
        #Iniciamos el timer encargado de generar más gatos
        self.timer_mas_gatos = QTimer()
        self.timer_mas_gatos.timeout.connect(self.mas_gatos)
        self.timer_mas_gatos.start(2000)

    def mostrar_clasificador(self, lista):
        lista_ordenada = sorted(lista, key=lambda x: x['puntaje'], reverse=True)
        ranking = "🏆 RANKING\n\n"
        for i, jugador in enumerate(lista_ordenada, start=1):
            ranking += f"{i}. {jugador['nombre']} - {jugador['puntaje']}\n"

        self.label_ranking.setText(ranking)

class Dino(QLabel):
    def __init__(self, ventana):
        super().__init__(ventana)

        #Cargo las 3 imagenes 
        self.sprites = [QPixmap(f"frontent/img/dino{num+1}.png") for num in range(3)]
        self.setPixmap(self.sprites[0])

        # Geometria del dinosaurio
        self.setGeometry(100,600,100,100)
        self.setScaledContents(True)
        self.imagen = 0

        # Thread encargado de cambiar el sprite
        self.thread_mover = QTimer()
        self.thread_mover.timeout.connect(self.cambiar_imagen)
        self.thread_mover.start(100)
        
        # Es False si perdemos
        self.seguimos = False

        # Lock para que no saltes 2 veces
        self.lock_salto = QMutex()



    def cambiar_imagen(self):
        # 10 veces por segundo, cambio la imagen de mi dinosaurio para que mueva
        # Las patitas
        if self.seguimos:
            self.imagen += 1
            self.setPixmap(self.sprites[self.imagen % 3])


    def saltar(self):
        # -----Explicacion------
        # Cada vez que apreto el espacio un thread se genera en el dinosaurio
        # El problema es que si 2 thread están haciendolo saltar simultaneamente
        # Entran en conflicto y el dinosaurio empieza a volar
        # Por eso se comprueba el QMutex con trylock, si es positivo hago saltar al dinosaurio
        # Sino simplemente paso de largo



        # Compruebo si el lock está pedido
        respuesta = self.lock_salto.tryLock()

        if respuesta:
            self.altura_salto = 300     
            self.velocidad = 8         
            self.subiendo = True        
            self.y_inicial = self.y()   
            self.timer_salto = QTimer()
            self.timer_salto.timeout.connect(self._animar_salto)
            self.timer_salto.start(10)
        else:
            return
        

    def _animar_salto(self):
        # se encarga de mover el dinosaurio hacia arriba hasta alcanzar la altura de salto
        # Luego lo hace bajar

        y_actual = self.y()
        
        if self.subiendo:
            nuevo_y = y_actual - self.velocidad
            if self.y_inicial - nuevo_y >= self.altura_salto:
                self.subiendo = False
        else:
            nuevo_y = y_actual + self.velocidad
            if nuevo_y >= self.y_inicial:
                nuevo_y = self.y_inicial
                self.lock_salto.unlock()
                self.timer_salto.stop()

        self.move(self.x(), nuevo_y)

class Gato(QLabel):
    def __init__(self, ventana):
        super().__init__(ventana)
        pixmap = QPixmap("frontent/img/gato.png")

        # La foto del gato mira hacia el otro lado, No habia otra en google :(
        # Así que la damos vuelta
        transform = QTransform().scale(-1, 1)
        pixmap_reflejado = pixmap.transformed(transform)

        # Seteamos el pixmap
        self.setPixmap(pixmap_reflejado)
        self.setGeometry(2000, 600, 100, 100)
        self.setScaledContents(True)
        self.mover = True


        #Thread que mueve el gato a la izquierda
        self.thread_mover = QTimer()
        self.thread_mover.timeout.connect(self.mover_izquierda)
        self.thread_mover.start(16)
        
        # Es False si perdemos
        self.seguimos = True
    
    def mover_izquierda(self):
        if self.seguimos:
            if self.mover:
                # Accedo a la posicion donde está
                pos = self.pos()
                pos_x = pos.x()
                pos_y = pos.y()

                # Muevo un poquito
                nuevo_x = pos_x - 8
                self.move(nuevo_x, pos_y)

            if nuevo_x <= -100:
                # Si tenemos que supera cierta barrera hacemos que se detenga
                self.seguimos = False
        else:
            # Si un gato está detenido borramos el thread que lo mueve
            self.thread_mover.stop()


# Backend

### ✅ Parte 5) Uso de networking con interfaz gráfica.

Debido a que contamos con un servidor y el servidor no es directamente una interacción con el usuario **EL BACKEND COMUNICA CON EL SERVIDOR**. Para evitar que el código se quede bloqueado cada vez que realizamos una request, asignamos un Qthread a cada request, cuando reciba una respuesta, el backend responde con un


In [ ]:
from PyQt5.QtCore import QObject, pyqtSignal, QThread
import requests

class Procesardor(QObject):
    senal_clasificador = pyqtSignal(list)

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

        # Asignamos una ip y puerto
        self.url = 'http://localhost:4631'
        self.variable_nombre = None

    def nombre(self, nombre):
        # Funcion que recibe el nombre que ingresó el usuario
        if self.variable_nombre is None:
            self.variable_nombre = nombre
            print(nombre)

    def obtener_clasificador(self):
        #-----Explicación-----
        # La libreria request hace que el codigo de python se quede
        # Esperando hasta recibir una respuesta, esto hace que se congele el juego
        # Solución: debemos tenerlo separado en otro thread

        thread = QThread()

        # Defino la función que maneja la libreria request       
        def tarea():
            try:
                respuesta = requests.get(self.url + "/puntajes", timeout=5)
                clasificador = respuesta.json()
                self.senal_clasificador.emit(clasificador)
            except Exception as e:
                print(f"Error en GET: {e}")
            thread.quit()

        thread.run = tarea
        thread.start()



    def perder(self, puntaje):
        # Misma idea que en Get pero realizando una solicitud post
        thread = QThread()

        def tarea():
            try:
                contenido = {
                    "nombre": self.variable_nombre,
                    "puntaje": puntaje
                }
                requests.post(self.url + "/puntajes", json=contenido, timeout=5)
            except Exception as e:
                print(f"Error en POST: {e}")
            finally:
                self.obtener_clasificador()
                thread.quit()
        thread.run = tarea
        thread.start()
